## Setup

In [ ]:
from google.colab import drive
# if you mount your drive somewhere else, change the path accordingly
drive.mount('/content/gdrive/')

import sys
#My Drive, change if needed
prefix = '/content/gdrive/My Drive/'
# modify "customized_path_to_your_homework" here to where you uploaded your homework
customized_path_to_your_homework = 'MachineLearning/Assignment1'
sys_path = prefix + customized_path_to_your_homework
sys.path.append(sys_path)

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


## XGBoost

In [ ]:
"""
Info:
NBA MVP Prediction with XGBoost
=================================
Labels: real MVP vote share from Player_Award_Shares.csv
Stats:  per-game features from Player_Totals.csv
Train: 2000–2022  |  Test: 2023–2025
"""

import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings("ignore")

#Load the award and player data
awards = pd.read_csv("gdrive/MyDrive/MachineLearning/Project/Player Award Shares.csv")
mvp_votes = awards[awards["award"] == "nba mvp"][
    ["season", "player_id", "share", "winner"]
].copy()
stats = pd.read_csv("gdrive/MyDrive/MachineLearning/Project/Player Totals.csv")
stats = stats[stats["lg"] == "NBA"].copy() #only for NBA data

#One edge case is players that get traded, we take only their season total and ignore their respective team totals
dup_mask = stats.duplicated(subset=["season", "player_id"], keep=False)
tot_mask = stats["team"] == "TOT"
stats = stats[~dup_mask | tot_mask].copy()

#take total and divide them by the number of games played to get ppg
#this allows for a more fair measurement, as a player with more games played may have higher totals
stats["ppg"] = stats["pts"] / stats["g"]
stats["rpg"] = stats["trb"] / stats["g"]
stats["apg"] = stats["ast"] / stats["g"]
stats["spg"] = stats["stl"] / stats["g"]
stats["bpg"] = stats["blk"] / stats["g"]
stats["topg"] = stats["tov"] / stats["g"]
stats["mpg"] = stats["mp"]  / stats["g"]
stats["ts_pct"] = stats["pts"] / (2 * (stats["fga"] + 0.44 * stats["fta"]) + 1e-6) #true shooting to measure effeciency
stats["start_rate"] = stats["gs"]  / stats["g"]

feature_cols = [
    "ppg", "rpg", "apg", "spg", "bpg", "topg", "mpg",
    "fg_percent", "ft_percent", "ts_pct", "start_rate", "trp_dbl", "g"
]

#Left join to merge the award data and our needed columns from the player data
df = mvp_votes.merge(stats[["season", "player_id", "player"] + feature_cols],
                     on=["season", "player_id"], how="left")
df = df.dropna(subset=feature_cols).copy()

print(f"Dataset: {len(df)} MVP candidate-seasons across "
      f"{df['season'].nunique()} seasons ({df['season'].min()}–{df['season'].max()})")
print(f"Features: {feature_cols}\n")

#prevent data leakage
train = df[df["season"].between(2000, 2022)]
test  = df[df["season"].between(2023, 2025)]

X_train, y_train = train[feature_cols], train["share"]
X_test,  y_test  = test[feature_cols],  test["share"]

#create the XGBoost
model = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0
)
model.fit(X_train, y_train)

#cross validation
cv = cross_val_score(model, X_train, y_train, cv=5,
                     scoring="neg_root_mean_squared_error")
print(f"5-Fold CV RMSE (vote share): {-cv.mean():.4f} ± {cv.std():.4f}")

test_preds = model.predict(X_test)
test_rmse = np.sqrt(np.mean((test_preds - y_test.values) ** 2))
print(f"Test RMSE (2023–2025): {test_rmse:.4f}\n")

#accuracy for per season rankings - assisted by claude
print("── Per-Season MVP Predictions ──")
top1_hits, top3_hits, total = 0, 0, 0

for season in sorted(test["season"].unique()):
    sdf = test[test["season"] == season].copy()
    sdf["pred"] = model.predict(sdf[feature_cols])
    sdf = sdf.sort_values("pred", ascending=False).reset_index(drop=True)
    sdf["pred_rank"] = sdf.index + 1

    winner_row = sdf[sdf["winner"] == True]
    actual_rank = int(winner_row["pred_rank"].values[0]) if len(winner_row) > 0 else None
    pred_name  = sdf.iloc[0]["player"]
    top3_names = sdf.head(3)["player"].tolist()

    hit1 = actual_rank == 1
    hit3 = actual_rank is not None and actual_rank <= 3
    flag = "✓" if hit1 else ("~" if hit3 else "✗")
    if hit1: top1_hits += 1
    if hit3: top3_hits += 1
    total += 1

    actual_name = winner_row["player"].values[0] if len(winner_row) > 0 else "?"
    print(f"  {season}  Predicted: {pred_name:<28} Actual MVP: {actual_name:<28} Rank: #{actual_rank}  {flag}")
    print(f"         Top 3: {top3_names}")
    # Show predicted vs actual vote share for top candidates
    print(f"         {'Player':<28} {'Actual Share':>13} {'Pred Share':>11}")
    for _, row in sdf.head(5).iterrows():
        print(f"         {row['player']:<28} {row['share']:>12.3f}  {row['pred']:>10.3f}")
    print()

print(f"Top-1 Accuracy: {top1_hits/total:.0%}   Top-3 Accuracy: {top3_hits/total:.0%}   ({total} seasons)\n")

#Determine how important a feature is in making the final decision - assisted by Claude
print("── Feature Importances ──")
imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
for feat, val in imp.items():
    bar = "█" * int(val * 80)
    print(f"  {feat:<15} {val:.4f}  {bar}")

#Prediction for current season - assisted by Claude
s26 = stats[stats["season"] == 2026].dropna(subset=feature_cols).copy()
s26["pred"] = model.predict(s26[feature_cols])
top5 = s26.sort_values("pred", ascending=False).head(5)
print("\n── 2026 MVP Race (Model Rankings) ──")
for i, (_, row) in enumerate(top5.iterrows(), 1):
    print(f"  #{i}  {row['player']:<28}  pred_share={row['pred']:.3f}"
          f"  ({row['ppg']:.1f}pts / {row['rpg']:.1f}reb / {row['apg']:.1f}ast)")

Dataset: 709 MVP candidate-seasons across 48 seasons (1978–2025)
Features: ['ppg', 'rpg', 'apg', 'spg', 'bpg', 'topg', 'mpg', 'fg_percent', 'ft_percent', 'ts_pct', 'start_rate', 'trp_dbl', 'g']

5-Fold CV RMSE (vote share): 0.2681 ± 0.0323
Test RMSE (2023–2025):       0.2442

── Per-Season MVP Predictions ──
  2023  Predicted: Giannis Antetokounmpo        Actual MVP: Joel Embiid                  Rank: #3  ~
         Top 3: ['Giannis Antetokounmpo', 'Nikola Jokić', 'Joel Embiid']
         Player                        Actual Share  Pred Share
         Giannis Antetokounmpo               0.606       0.770
         Nikola Jokić                        0.674       0.732
         Joel Embiid                         0.915       0.684
         Luka Dončić                         0.010       0.571
         Jayson Tatum                        0.280       0.459

  2024  Predicted: Giannis Antetokounmpo        Actual MVP: Nikola Jokić                 Rank: #3  ~
         Top 3: ['Giannis Antetokou